# 📷 PHOTO LAB HQ — 自分の被写体を覚えさせて、新しい画像を作る

ペット・人・モノなど **1つの被写体** の写真を10〜30枚用意すると、その被写体を覚えた画像生成AIを作れます。
「海辺にいる◯◯」「水彩画の◯◯」のように、文章で指示した場面の画像を生成できます。

- 計算は **Google Colab の無料GPU（T4）** で行います。お金はかかりません。
- ベースは公開モデル **Stable Diffusion XL**。そこに写真から学んだ差分（LoRA）を足します。
- 写真・学習結果は **あなたのGoogleドライブとこのColab** の中だけに置かれます。

### 使い方
1. 上のメニュー **「ランタイム」→「ランタイムのタイプを変更」→「T4 GPU」** になっているか確認
2. 下のセルを **上から順に ▶ で実行**（各セルの右側のフォームで設定を変えられます）
3. 学習は写真20枚で **40〜60分** ほど。終わったら「05 生成」で好きな画像を作れます
4. 次回からは **01 → 02 → 05** だけ実行すれば、学習済みの被写体ですぐ生成できます（学習結果はドライブに残ります）

> 無料GPUは混雑時に使えないことや、長時間放置で切断されることがあります。学習の途中経過はドライブに保存されるので、切れたら同じセルをもう一度実行すれば続きから再開します。

In [ ]:
#@title 01 準備（GPU確認・ライブラリのインストール）　約3分
import subprocess, sys
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True)
if gpu.returncode != 0:
    raise SystemExit("❌ GPUが有効になっていません。「ランタイム」→「ランタイムのタイプを変更」で T4 GPU を選んでから、もう一度実行してください。")
print("GPU:", gpu.stdout.strip())

DIFFUSERS_VERSION = "0.42.1"
!pip -q install "diffusers=={DIFFUSERS_VERSION}" "peft>=0.17" "accelerate>=1.0" "transformers>=4.45" bitsandbytes pillow-heif ftfy tensorboard
!wget -q -O /content/train_dreambooth_lora_sdxl.py https://raw.githubusercontent.com/huggingface/diffusers/v{DIFFUSERS_VERSION}/examples/dreambooth/train_dreambooth_lora_sdxl.py
!accelerate config default > /dev/null 2>&1
print("✅ 準備完了")

In [ ]:
#@title 02 写真を用意する
#@markdown **SOURCE**：`drive` はGoogleドライブのフォルダから読み込み（iPhoneなら「Googleドライブ」アプリで写真をそのフォルダに入れるのが簡単）。`upload` はこの画面から直接アップロード。
SOURCE = "drive" #@param ["drive", "upload"]
#@markdown ドライブ内のフォルダ（マイドライブ直下からのパス）。学習結果もこのフォルダの中に保存します。
DRIVE_FOLDER = "photolab/subject" #@param {type:"string"}
#@markdown 学習の解像度。1024がきれい、768は速い。
RESOLUTION = 1024 #@param [1024, 768] {type:"raw"}

import os, glob, shutil
from pathlib import Path
from PIL import Image, ImageOps
import pillow_heif
pillow_heif.register_heif_opener()  # HEIC/HEIF をそのまま開けるようにする

if SOURCE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = Path("/content/drive/MyDrive") / DRIVE_FOLDER
    WORK.mkdir(parents=True, exist_ok=True)
    src_files = [p for p in WORK.iterdir() if p.is_file()]
    if not src_files:
        raise SystemExit(f"❌ {WORK} に写真がありません。ドライブのこのフォルダに写真を入れてから、もう一度実行してください。")
else:
    from google.colab import files
    WORK = Path("/content/photolab"); up = WORK / "upload"; up.mkdir(parents=True, exist_ok=True)
    print("写真を選んでください（複数可・HEICも可）")
    for name, data in files.upload().items():
        (up / name).write_bytes(data)
    src_files = list(up.iterdir())

OUT = WORK / "_lora"            # 学習結果（途中経過も含む）
GEN = WORK / "_generated"       # 生成した画像
DATA = Path("/content/instance")  # 学習用に整えた写真
shutil.rmtree(DATA, ignore_errors=True); DATA.mkdir(parents=True)

def prepare(path, size):
    """向きを直して中央を正方形に切り抜き、学習サイズに縮小する"""
    im = ImageOps.exif_transpose(Image.open(path)).convert("RGB")
    s = min(im.size); l, t = (im.width - s) // 2, (im.height - s) // 2
    return im.crop((l, t, l + s, t + s)).resize((size, size), Image.LANCZOS)

ok, ng = 0, []
for p in sorted(src_files):
    if p.name.startswith("."): continue
    try:
        prepare(p, RESOLUTION).save(DATA / f"{ok:03d}.png"); ok += 1
    except Exception as e:
        ng.append(p.name)
print(f"✅ {ok}枚を準備しました" + (f"（読めなかったファイル: {', '.join(ng[:10])}）" if ng else ""))
if ok < 5: print("⚠️ 写真が少なめです。被写体が写った写真を10〜30枚（いろいろな角度・背景）にすると良くなります。")
if ok > 60: print("ℹ️ 枚数が多いので学習に時間がかかります。似た写真を間引くと速くなります。")

import matplotlib.pyplot as plt
show = sorted(DATA.glob("*.png"))[:12]
fig, axes = plt.subplots(1, len(show), figsize=(2 * len(show), 2))
for ax, p in zip(axes if len(show) > 1 else [axes], show):
    ax.imshow(Image.open(p)); ax.axis("off")
plt.show()

In [ ]:
#@title 03 被写体の設定
#@markdown 被写体の種類を英語の1単語で（例：`dog` `cat` `person` `man` `woman` `toy` `car` `plant` `bag`）
CLASS_WORD = "dog" #@param {type:"string"}
#@markdown 被写体の「名前」になる合言葉。普通の単語とかぶらない文字列にします（そのままでOK）。
TOKEN = "ohwx" #@param {type:"string"}
SUBJECT = f"{TOKEN} {CLASS_WORD}".strip()
INSTANCE_PROMPT = f"a photo of {SUBJECT}"
print("学習で使う文：", INSTANCE_PROMPT)
print("生成するときは文の中に「" + SUBJECT + "」を入れてください。")

In [ ]:
#@title 04 学習する　写真20枚で40〜60分
#@markdown 学習回数。`0` なら枚数から自動で決めます（1枚あたり50回、500〜1000回）。
STEPS = 0 #@param {type:"integer"}
#@markdown 覚える細かさ（大きいほど細部まで覚えるが、似すぎやすい）
RANK = 16 #@param [8, 16, 32] {type:"raw"}
LEARNING_RATE = 1e-4 #@param {type:"number"}

import subprocess, json, time
n = len(list(DATA.glob("*.png")))
steps = STEPS if STEPS > 0 else max(500, min(1000, n * 50))
OUT.mkdir(parents=True, exist_ok=True)
resume = ["--resume_from_checkpoint", "latest"] if list(OUT.glob("checkpoint-*")) else []
if resume: print("↻ 前回の途中経過から再開します")

cmd = [
    "accelerate", "launch", "/content/train_dreambooth_lora_sdxl.py",
    "--pretrained_model_name_or_path", "stabilityai/stable-diffusion-xl-base-1.0",
    "--pretrained_vae_model_name_or_path", "madebyollin/sdxl-vae-fp16-fix",  # fp16でも色が崩れないVAE
    "--instance_data_dir", str(DATA),
    "--instance_prompt", INSTANCE_PROMPT,
    "--output_dir", str(OUT),
    "--resolution", str(RESOLUTION),
    "--train_batch_size", "1",
    "--gradient_accumulation_steps", "1",
    "--gradient_checkpointing",
    "--use_8bit_adam",
    "--mixed_precision", "fp16",
    "--learning_rate", str(LEARNING_RATE),
    "--lr_scheduler", "constant",
    "--lr_warmup_steps", "0",
    "--max_train_steps", str(steps),
    "--rank", str(RANK),
    "--random_flip",
    "--checkpointing_steps", "200",
    "--checkpoints_total_limit", "1",
    "--report_to", "tensorboard",
    "--seed", "0",
] + resume
print(f"学習 {steps} 回・{n}枚・{RESOLUTION}px を開始します（最初にベースモデル約7GBをダウンロードします）\n")
t0 = time.time()
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    # 進捗バーと重要な行だけ表示する
    if "Steps:" in line or "Error" in line or "error" in line or "Traceback" in line or "Saved" in line or "OutOfMemory" in line:
        print(line.rstrip()[-160:], flush=True)
proc.wait()
weights = OUT / "pytorch_lora_weights.safetensors"
if proc.returncode != 0 and not (weights.exists() and weights.stat().st_mtime > t0):
    raise SystemExit("❌ 学習が途中で止まりました。もう一度このセルを実行すると途中から再開します。メモリ不足と出た場合は 02 で RESOLUTION を 768 にして、02 から実行し直してください。")
# ※ 学習後の確認用の画像生成で止まっても、学習結果は保存済みなので問題ありません
json.dump({"subject": SUBJECT, "class": CLASS_WORD, "token": TOKEN, "resolution": RESOLUTION, "steps": steps, "images": n},
          open(OUT / "photolab.json", "w"), ensure_ascii=False)
for c in OUT.glob("checkpoint-*"): shutil.rmtree(c, ignore_errors=True)  # 完了したので途中経過は消す
print(f"\n✅ 学習完了（{(time.time() - t0) / 60:.0f}分）。結果: {OUT / 'pytorch_lora_weights.safetensors'}")

In [ ]:
#@title 05 生成する（何度でも実行できます）
#@markdown 生成したい場面を英語で。`{s}` の部分が被写体に置き換わります。
PROMPT = "a photo of {s} sitting on a beach at sunset, golden hour, 35mm film" #@param {type:"string"}
NEGATIVE = "blurry, low quality, deformed, extra limbs, watermark, text" #@param {type:"string"}
NUM_IMAGES = 4 #@param {type:"slider", min:1, max:8, step:1}
#@markdown 被写体への似せ具合（LoRAの強さ）。似ないなら上げる、崩れるなら下げる。
LORA_SCALE = 0.9 #@param {type:"slider", min:0.3, max:1.3, step:0.05}
STEPS_GEN = 30 #@param {type:"slider", min:15, max:50, step:5}
SEED = -1 #@param {type:"integer"}

import torch, json, random, datetime
from diffusers import StableDiffusionXLPipeline, AutoencoderKL
from pathlib import Path

meta = json.load(open(OUT / "photolab.json"))
if "pipe" not in globals():
    vae = AutoencoderKL.from_pretrained("madebyollin/sdxl-vae-fp16-fix", torch_dtype=torch.float16)
    pipe = StableDiffusionXLPipeline.from_pretrained("stabilityai/stable-diffusion-xl-base-1.0", vae=vae,
                                                     torch_dtype=torch.float16, variant="fp16", use_safetensors=True).to("cuda")
    pipe.load_lora_weights(str(OUT), weight_name="pytorch_lora_weights.safetensors", adapter_name="subject")
    pipe.set_progress_bar_config(disable=True)
pipe.set_adapters(["subject"], adapter_weights=[LORA_SCALE])

prompt = PROMPT.replace("{s}", meta["subject"])
if meta["token"] not in prompt:
    prompt = f"{meta['subject']}, {prompt}"
seed = SEED if SEED >= 0 else random.randint(0, 2**31 - 1)
print("PROMPT:", prompt, "| SEED:", seed)
GEN.mkdir(parents=True, exist_ok=True)
stamp = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
images = []
for i in range(NUM_IMAGES):
    g = torch.Generator("cuda").manual_seed(seed + i)
    img = pipe(prompt, negative_prompt=NEGATIVE, num_inference_steps=STEPS_GEN, guidance_scale=6.0,
               width=1024, height=1024, generator=g).images[0]
    img.save(GEN / f"{stamp}-{seed + i}.png"); images.append(img)

import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, len(images), figsize=(5 * len(images), 5))
for ax, im in zip(axes if len(images) > 1 else [axes], images):
    ax.imshow(im); ax.axis("off")
plt.show()
print(f"💾 保存先: {GEN}")

### 🎨 プロンプトの例（`{s}` が被写体）
- `a photo of {s} in a snowy forest, soft morning light`
- `{s} wearing a tiny astronaut suit, studio photo`
- `watercolor painting of {s}, pastel colors`
- `{s} as a lego figure, product photo`
- `a polaroid photo of {s} in a Tokyo cafe`

うまくいかないとき：**似ていない** → LORA_SCALE を上げる／04 で STEPS を増やす　・　**背景や構図がいつも同じ** → LORA_SCALE を下げる／写真のバリエーションを増やす

In [ ]:
#@title 06（おまけ）スマホのブラウザから生成する（先に 05 を1回実行してください）
#@markdown 実行すると `https://xxxx.gradio.live` のリンクが出ます。iPhoneで開くと、文章を入れて生成できます。
#@markdown ⚠️ そのリンクを知っている人は誰でも使えます。使い終わったらこのセルを停止してください（最長72時間で自動的に無効になります）。
!pip -q install gradio
import gradio as gr, torch, random, json

meta = json.load(open(OUT / "photolab.json"))
def make(prompt, scale, seed):
    pipe.set_adapters(["subject"], adapter_weights=[scale])
    p = prompt.replace("{s}", meta["subject"])
    if meta["token"] not in p: p = f"{meta['subject']}, {p}"
    seed = int(seed) if seed >= 0 else random.randint(0, 2**31 - 1)
    img = pipe(p, negative_prompt=NEGATIVE, num_inference_steps=STEPS_GEN, guidance_scale=6.0,
               generator=torch.Generator("cuda").manual_seed(seed)).images[0]
    GEN.mkdir(parents=True, exist_ok=True); img.save(GEN / f"web-{seed}.png")
    return img, f"seed {seed}"

with gr.Blocks(title="PHOTO LAB HQ") as ui:
    gr.Markdown(f"## PHOTO LAB HQ\n被写体：**{meta['subject']}**　文中の `{{s}}` が被写体になります")
    prompt = gr.Textbox(value="a photo of {s} on a beach at sunset", label="PROMPT")
    with gr.Row():
        scale = gr.Slider(0.3, 1.3, value=0.9, step=0.05, label="似せ具合")
        seed = gr.Number(value=-1, label="SEED（-1でランダム）")
    btn = gr.Button("生成", variant="primary")
    out, info = gr.Image(label="結果"), gr.Markdown()
    btn.click(make, [prompt, scale, seed], [out, info])
ui.launch(share=True, debug=True)